# TSFM RUL benchmark: thin orchestration notebook

**Run all** executes the full campaign — every dataset in `src/datasets/` × every TSFM in
`src/models/` — via `src/campaign.py`: Stage A cache → data-scaling sweep → fairness arms →
horizon eval → saved figures, per combo, all restartable. Datasets not downloaded into
`Data/` are skipped with a notice. Single-dataset deep-dives (ablation, raised label cap,
transfer, significance tables) are gated behind `RUN_DEEP_DIVES` in the Config cell.

## 1. Setup

In [ ]:
# Pinned installs. Colab already ships torch/numpy/pandas/scikit-learn/matplotlib.
# chronos-forecasting 2.x provides the official Chronos2Pipeline.embed().
%pip install -q "chronos-forecasting>=2.0.0" "coral-pytorch==1.4.0" "lightgbm>=4.0" "sktime>=1.0" "numba>=0.59" "safetensors>=0.4"

import sys, os, torch
from google.colab import drive

# Mount Drive (holds CMAPSSData/, the embedding cache, and results).
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Add the repo (kept on Drive) to sys.path so `import src.*` works.
REPO_DIR = '/content/drive/MyDrive/Predictive-Maintenance-LSTM'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Config

The single source of truth is `src/config.py`. The values below are the recorded FD001
ablation winner (CHANGES.md §12) — the campaign runs every dataset/model at this config.
`sensor_columns` resolves per dataset automatically (CHANGES.md §24); results and figures
are auto-named `<dataset>_<model>_…` by the campaign, so nothing collides.

**Drop the raw datasets under one `data_root`, one subfolder per family:**

```
Data/
  CMAPSSData/     FD001–FD004 text files (committed)
  XJTU-SY/        35Hz12kN/  37.5Hz11kN/  40Hz10kN/   (each holds BearingC_B/ folders)
  N-CMAPSS/       N-CMAPSS_DS01-005.h5 … N-CMAPSS_DS08d-010.h5   (all .h5 files, flat)
```

The XJTU folder may instead be named `XJTU-SY_Bearing_Datasets` (the zip's own name) and
may have one extra nesting level — both load without renaming (CHANGES.md §26). The first
N-CMAPSS run parses each `.h5` once into a per-file aggregate cache under `cache_dir`
(minutes each; cached thereafter, CHANGES.md §27). Whatever isn't downloaded is skipped.

In [ ]:
from src.config import Config

DRIVE = '/content/drive/MyDrive/Predictive Maintenance LSTM'
RUN_DEEP_DIVES = False     # True => also run the single-dataset deep-dive sections below

config = Config(
    data_root=f'{DRIVE}/Data',        # Data/CMAPSSData, Data/XJTU-SY, Data/N-CMAPSS (see above)
    cache_dir=f'{DRIVE}/cache',
    results_dir=f'{DRIVE}/results',
    # --- the recorded ablation winner (CHANGES.md §12); the campaign uses these ---
    tsfm_context_length=256,
    head_features='emb+locscale',
    pooling='mean',
    # experiment_name='v2',            # optional extra prefix on top of <dataset>_<model>
    # embed_batch_size=64,             # lower for a T4 (esp. N-CMAPSS: 37 channels)
    # losses=['mse', 'corn', 'quantile'],
)
config

## 3. Campaign — all datasets × all TSFMs (the Run-all path)

`run_campaign` sweeps `datasets.all_dataset_names()` × `models.EMBEDDERS` (CHANGES.md §24):
**FD001–FD004 + XJTU-SY + N-CMAPSS DS01…DS08d + DSALL** (the combined N-CMAPSS fleet, the
RQ1 high-data arm — CHANGES.md §28). Per combo (restartable, resumable): Stage A cache →
sweep → fairness arms → horizon eval → figures. Missing datasets are skipped with a notice;
a failing combo is reported and the campaign continues. Every artifact is named
`<dataset>_<model>_…`.

Per-dataset protocol choices are recorded once in `campaign.DEFAULT_DATASET_OVERRIDES`
(CHANGES.md §30): XJTU-SY cycles are MINUTES (so its `max_rul`/`window_size` are pinned)
and DSALL's member list is pinned for a deterministic cache key. Passing
`dataset_overrides=None` (the default) uses those; a dict merges over them (you win per
key); an explicit `{}` opts out entirely.

In [ ]:
import torch
from src.campaign import run_campaign, DEFAULT_DATASET_OVERRIDES

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DEFAULT_DATASET_OVERRIDES   # inspect the recorded per-dataset protocol choices

# dataset_overrides=None (default) => the recorded DEFAULT_DATASET_OVERRIDES.
# To tweak one dataset without losing the rest, pass a dict (merged over the defaults):
#   dataset_overrides={'XJTU-SY': {'max_rul': 90}}   # keeps window_size=30, ctx=256
summary = run_campaign(
    config,
    device=device,
)
summary

## 4. Deep-dives (single dataset; set `RUN_DEEP_DIVES = True` to include in Run-all)

Everything below reproduces the focused FD001 studies: the ablation that picked the config,
learning curves, the CORN-vs-MSE significance table, the raised-label-cap arm, and the
cold-start transfer. Each cell no-ops when `RUN_DEEP_DIVES` is False.

## 3b. Stage A2 — Ablation (pick the winning config)

Full-data, MSE, 3 seeds over `tsfm_context_length ∈ {30,60,120,256} × head_features ∈ {emb, emb+locscale}`, then the `emb+locscale+raw` arm and the `{mean, last_content}` pooling variants at the best cell. Each (context, pooling) triggers one Stage A cache build (idempotent). Restartable — rows are checkpointed to `ablation.csv`. `select_best_ablation_cell` picks the winner by **clipped RMSE** (the literature-comparable metric). Record the winner + justification in `CHANGES.md` §9.

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.sweep import run_ablation, select_best_ablation_cell
    import torch, pandas as pd

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ablation_csv = run_ablation(config, device=device)   # builds caches + trains heads over the grid

    best = select_best_ablation_cell(ablation_csv)        # winner by clipped RMSE
    print('WINNING CELL:', best)

    # Full ablation table (mean ± std clipped RMSE over seeds), sorted best-first.
    abl = pd.read_csv(ablation_csv)
    summary = (abl.groupby(['tsfm_context_length', 'head_features', 'pooling'])['rmse_clipped']
                  .agg(['mean', 'std', 'count']).sort_values('mean'))
    summary


### Sweep + window comparison at an explicit config (superseded by the campaign for standard runs)

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.sweep import run_sweep, run_baseline_window_comparison
    import torch

    device = 'cuda' if torch.cuda.is_available() else 'cpu'   # heads train on-GPU (tensor slicing)

    # Adopt the ablation winner. (Also builds its Stage A cache if not already present.)
    sweep_config = config.replace(
        tsfm_context_length=int(best['tsfm_context_length']),
        head_features=best['head_features'],
        pooling=best['pooling'],
    )
    from src.embeddings import build_embedding_cache
    build_embedding_cache(sweep_config)                       # idempotent; no-op if Stage A2 built it

    # Optional (Task 1.5): compare GBM/LSTM at window 30 vs 120 at full data. If a longer
    # window wins, adopt it per baseline, e.g. sweep_config = sweep_config.replace(
    #     baseline_windows={'lstm': 120}).  Then baselines are re-windowed from the raw series.
    # run_baseline_window_comparison(config, device=device)

    results_csv = run_sweep(sweep_config, device=device)
    print('results CSV:', results_csv)


### 4b. Fairness arms — hand the baselines the age signal (CHANGES.md §19)

The winning TSFM config sees up to 256 cycles while baselines see 30, and its
variable-length context implicitly carries engine age (CHANGES.md §12 caveat 2).
These arms bound how much of the gap that explains:
- `run_baseline_window_comparison`: GBM/LSTM at windows {30, 60, 120} — if a longer
  window wins at full data, adopt it via `sweep_config.replace(baseline_windows=...)`
  and rerun the sweep cell.
- `run_fairness_baselines`: the `cycle_reg` age floor (plan §4) + `gbm_age` (GBM with
  `time_cycles` as an extra channel). Rows land in `results_v2.csv`, so the
  data-scaling figure picks them up automatically.

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.sweep import run_baseline_window_comparison, run_fairness_baselines
    import pandas as pd

    bw_csv = run_baseline_window_comparison(sweep_config, windows=[30, 60, 120], device=device)
    print(pd.read_csv(bw_csv).groupby(['model', 'baseline_window'])['rmse_clipped']
            .agg(['mean', 'std']).round(2))
    # If a longer window clearly wins for a baseline, adopt it and rerun the sweep:
    # sweep_config = sweep_config.replace(baseline_windows={'lstm': 120}); run_sweep(...)

    run_fairness_baselines(sweep_config)   # cycle_reg + gbm_age -> results_v2.csv (CPU)


## 5. Stage C — Results

### Data-scaling curve (the headline figure): test error vs. training units, one line per model, error bands over seeds. Plotted for the **clipped** protocol (literature-comparable) and the **unclipped** protocol.

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    # Stage C figures (src/plots.py): saved to <results_dir>/figures/ as 300-dpi PNG + PDF.
    # predict_mean is a gray reference line (no longer squashes the y-axis); NASA panels are log-y.
    from pathlib import Path
    from src.plots import plot_data_scaling, plot_ablation

    figures_dir = config.figures_dir()
    saved = plot_data_scaling(results_csv, figures_dir, prefix=config.result_prefix())
    saved += plot_ablation(ablation_csv, figures_dir, prefix=config.result_prefix())
    print('saved:', *[str(s) for s in saved], sep='\n  ')


### Learning curves: validation RMSE vs. epoch, per sweep cell.

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    # Learning curves: one panel per loss arm, shaded by training-unit count (darker = more units).
    from src.plots import plot_learning_curves

    saved = plot_learning_curves(config.results_path('runs') / 'learning_curves', figures_dir, prefix=config.result_prefix())
    print('saved:', *[str(s) for s in saved], sep='\n  ')


## 6. Horizon-stratified evaluation — error vs. distance-to-failure

The main protocol scores one prediction per test unit (its last cycle), so far-from-failure
behavior — the predictions that buy planning time — is invisible. This embeds EVERY test
cycle (same context construction as training rows, separate cache; the main cache stays
valid) and stratifies error by the unclipped true RUL.

**Reading it:** bins below `max_rul` measure real accuracy at that horizon; the `≥125` bin
only measures saturation quality (training labels are clipped, so no model can express
horizons beyond the cap — see `src/horizon.py` docstring / CHANGES.md §16).

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.horizon import build_horizon_cache, run_horizon_eval

    build_horizon_cache(sweep_config)   # GPU: embeds every test cycle once (idempotent)
    horizon_csv = run_horizon_eval(sweep_config, n_units_list=[10, 100], device=device)
    print('horizon CSV:', horizon_csv)


In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.plots import plot_horizon, plot_horizon_trajectories

    saved = plot_horizon(horizon_csv, figures_dir, prefix=sweep_config.result_prefix())
    saved += plot_horizon_trajectories(
        sweep_config.results_path('horizon_predictions.csv'), figures_dir,
        max_rul=sweep_config.max_rul, prefix=sweep_config.result_prefix())
    print('saved:', *[str(s) for s in saved], sep='\n  ')


### 6a. CORN vs MSE per horizon bin — paired significance (CHANGES.md §18)

The horizon eval now defaults to all 5 seeds. Pairing by seed is valid (both loss arms
share each seed's sampled units and split). With 5 seeds the test is low-powered —
read p-values as descriptive, alongside the per-bin means.

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.evaluate import paired_seed_ttest
    import pandas as pd

    sig = pd.DataFrame(paired_seed_ttest(horizon_csv, loss_a='corn', loss_b='mse',
                                         metric='mae_clipped'))
    sig[['mean_corn', 'mean_mse', 'mean_delta', 't', 'p']] = \
        sig[['mean_corn', 'mean_mse', 'mean_delta', 't', 'p']].round(3)
    sig  # mean_delta < 0 => CORN better in that bin


## 6b. Raised label cap — genuine long-horizon skill (max_rul = 200)

The 125-cap runs above stay untouched (literature-comparable). This arm re-keys the
caches (labels are cached with the windows, so it is a fresh Stage A pass) and re-runs
the horizon eval at `max_rul=200`; bins extend automatically to {125–150, 150–175,
175–200, ≥200}. Both cap arms share `horizon.csv` / `horizon_predictions.csv`
(`max_rul` is part of the row key). Bins **below 125 are directly comparable across
arms** — if the 200-cap model gets worse there, the extra horizon costs near-failure
accuracy; the 125–200 bins measure whether degradation is detectable that early at all
(CHANGES.md §18).

NOTE: if a pre-existing `horizon_predictions.csv` predates the `max_rul` column, the
run stops with a schema error — move the old file into e.g. `results/archive/` first.

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    mr200_config = sweep_config.replace(max_rul=200)

    build_embedding_cache(mr200_config)      # fresh Stage A pass (new cache key)
    build_horizon_cache(mr200_config)        # GPU: every test cycle at the new key
    horizon200_csv = run_horizon_eval(mr200_config, n_units_list=[10, 100], device=device)
    print('horizon CSV (both cap arms):', horizon200_csv)


In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    saved = plot_horizon(horizon200_csv, figures_dir, prefix=mr200_config.result_prefix())   # one figure per (cap, n_units)
    saved += plot_horizon_trajectories(
        mr200_config.results_path('horizon_predictions.csv'), figures_dir,
        max_rul=200, prefix=mr200_config.result_prefix())
    print('saved:', *[str(s) for s in saved], sep='\n  ')


## 7. Cold-start transfer — deploy with zero target failures

Can a head trained on an analogous fleet (source) produce useful RUL on a fleet with **no
recorded failures** (target), and how fast do a few target failures close the gap? Three
arms per shot count: `zero_shot` (source-only), `target_only`, `source+target`
(CHANGES.md §17). FD001↔FD003 is the a-priori-valid pair (both single-condition, same
non-constant sensors); FD002/FD004 pairs are now supported through condition-wise
normalization, fit per dataset on its own train split (CHANGES.md §21). Builds the target's Stage A cache on first run (GPU).

In [ ]:
if RUN_DEEP_DIVES:   # gated: set True in the Config cell (§2)
    from src.transfer import run_transfer_eval
    from src.plots import plot_transfer

    transfer_csv = run_transfer_eval(sweep_config, source_dataset='FD001',
                                     target_dataset='FD003',
                                     shots=[2, 5, 10, 25], device=device)
    saved = plot_transfer(transfer_csv, figures_dir, prefix=sweep_config.result_prefix())
    print('saved:', *[str(s) for s in saved], sep='\n  ')
